# 18 Python intermediate - `logging` v2.0

_Kamil Bartocha_

## Rozkład jazdy

1. 🔹 **Poziomy logowania i podstawowa konfiguracja** - `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`
2. 🔹 **Handlery i formattery** - `FileHandler`, `StreamHandler`, `Formatter`
3. 🔹 **Loggery hierarchiczne i dobre praktyki** - `getLogger(__name__)`, propagacja

🐍 Każda sekcja zawiera ćwiczenia.

---

## 1. 🔹 Poziomy logowania i podstawowa konfiguracja

Moduł `logging` to standardowy mechanizm rejestrowania zdarzeń
w aplikacjach. Zastępuje `print()` - logi mają poziomy wazności,
mozna je wlaczac/wylaczac i kierowac do roznych miejsc.

| Poziom | Wartość | Kiedy uzywac |
|---|---|---|
| `DEBUG` | 10 | Szczegóły diagnostyczne, tylko przy debugowaniu |
| `INFO` | 20 | Potwierdzenie normalnego działania |
| `WARNING` | 30 | Cos nieoczekiwanego, program działa dalej |
| `ERROR` | 40 | Poważny błąd, część funkcjonalności niedostepna |
| `CRITICAL` | 50 | Błąd krytyczny, program moze nie działac |

Domyślny poziom root loggera to `WARNING` - logi ponizej nie sa
wyswietlane bez konfiguracji.

In [ ]:
import logging

# basicConfig - najprostsza konfiguracja
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)s - %(message)s'
)

logging.debug('Szczegoły diagnostyczne')     # DEBUG
logging.info('Aplikacja wystartowała')        # INFO
logging.warning('Mało miejsca na dysku')      # WARNING
logging.error('Nie mozna otworzyc pliku')     # ERROR
logging.critical('Utracono połączenie z DB')  # CRITICAL

In [ ]:
import logging

# Format z czasem, nazwa loggera i numerem linii
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(name)s:%(lineno)d - %(message)s',
    datefmt='%H:%M:%S'
)

# Logowanie z argumentami (lazy formatting - wydajniejsze niz f-string)
user_id = 42
logging.info('Uzytkownik %s zalogował sie', user_id)

# Logowanie wyjatku ze stack trace
try:
    x = 1 / 0
except ZeroDivisionError:
    logging.exception('Nieoczekiwany błąd podczas obliczen')

---

### 🐍 Ćwiczenia - poziomy i konfiguracja

**Ćw. 1.** Skonfiguruj `basicConfig` z poziomem `DEBUG` i formatem
zawierającym czas (`%(asctime)s`), poziom i wiadomość.
Zaloguj po jednym komunikacie na każdym z 5 poziomów.

**Ćw. 2.** Napisz funkcję `safe_divide(a, b)` zwracającą `a / b`.
Jeśli `b == 0`, zaloguj `ERROR` z wartościami `a` i `b` i zwróć `None`.
Jeśli wynik > 1000, zaloguj `WARNING`. W normalnym przypadku `DEBUG`.

**Ćw. 3. *(Trudniejsze)*** Napisz dekorator `log_calls(logger)`,
który loguje na poziomie `DEBUG` nazwę funkcji, argumenty i wynik
każdego wywołania. Obsługuj też wyjątki (loguj na `ERROR`).

In [ ]:
# Ćw. 1
import logging

logging.basicConfig(
    level=...,
    format=...
)

logging.debug(...)
logging.info(...)
logging.warning(...)
logging.error(...)
logging.critical(...)

In [ ]:
# Ćw. 2
import logging

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(message)s')

def safe_divide(a, b):
    ...


print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None, log ERROR
print(safe_divide(2000, 1))  # 2000.0, log WARNING

In [ ]:
# Ćw. 3
import logging
from functools import wraps

def log_calls(logger):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # hint: loguj przed wywolaniem i po, przechwytuj wyjatki
            ...
        return wrapper
    return decorator


logger = logging.getLogger('myapp')
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(message)s')

@log_calls(logger)
def add(a, b):
    return a + b

print(add(2, 3))   # log: calling add(2, 3) -> 5

---

## 2. 🔹 Handlery i formattery

Handler (handler) decyduje dokąd trafia log. Mozna ich podłączyc kilka.
Formatter (formatter) określa wygląd wiadomości.

| Handler | Cel |
|---|---|
| `StreamHandler` | konsola (stdout/stderr) |
| `FileHandler` | plik tekstowy |
| `RotatingFileHandler` | plik z rotacją po rozmiarze |
| `TimedRotatingFileHandler` | plik z rotacją czasową |
| `NullHandler` | ignorowanie (dla bibliotek) |

In [ ]:
import logging

# Tworzenie loggera z handlerem do pliku i konsoli
logger = logging.getLogger('app')
logger.setLevel(logging.DEBUG)

# Handler do konsoli - tylko WARNING i powyzej
console = logging.StreamHandler()
console.setLevel(logging.WARNING)
console.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))

# Handler do pliku - wszystko od DEBUG
file_h = logging.FileHandler('app.log', encoding='utf-8')
file_h.setLevel(logging.DEBUG)
file_h.setFormatter(logging.Formatter(
    '%(asctime)s [%(levelname)-8s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))

logger.addHandler(console)
logger.addHandler(file_h)

logger.debug('To tylko w pliku')
logger.warning('To pojawi sie na konsoli i w pliku')
logger.error('Blad!')

# Sprzatanie
for h in logger.handlers[:]:
    h.close()
    logger.removeHandler(h)
import os; os.remove('app.log') if os.path.exists('app.log') else None

---

### 🐍 Ćwiczenia - handlery i formattery

**Ćw. 4.** Stwórz logger `'audit'` z dwoma handlerami:
- `StreamHandler` na poziomie `INFO` z formatem `[INFO] message`
- `FileHandler` (`audit.log`) na poziomie `DEBUG` z datą i godziną.
Zaloguj 3 komunikaty na różnych poziomach i sprawdź różnicę.

**Ćw. 5.** Napisz funkcję `get_logger(name, log_file)` tworzącą i
zwracającą skonfigurowany logger - konsola `WARNING+`, plik `DEBUG+`.
Ustaw `propagate=False` by uniknąc duplikatów w root loggerze.

**Ćw. 6. *(Trudniejsze)*** Napisz własny handler `InMemoryHandler`
dziedziczący po `logging.Handler`. Przechowuje ostatnie `n` rekordów
w liście. Metoda `get_records()` zwraca sformatowane wiadomości.

In [ ]:
# Ćw. 4
import logging

audit = logging.getLogger('audit')
audit.setLevel(logging.DEBUG)

console_h = ...
file_h = ...

audit.addHandler(console_h)
audit.addHandler(file_h)

audit.debug('Szczegol')
audit.info('Uzytkownik zalogowany')
audit.warning('Niepoprawne haslo')

# sprzatanie
for h in audit.handlers[:]:
    h.close(); audit.removeHandler(h)
import os; os.remove('audit.log') if os.path.exists('audit.log') else None

In [ ]:
# Ćw. 5
import logging

def get_logger(name: str, log_file: str) -> logging.Logger:
    ...


logger = get_logger('mymodule', 'mymodule.log')
logger.debug('debug msg')
logger.warning('warning msg')
import os; os.remove('mymodule.log') if os.path.exists('mymodule.log') else None

In [ ]:
# Ćw. 6
import logging
from collections import deque

class InMemoryHandler(logging.Handler):
    def __init__(self, capacity: int = 100):
        super().__init__()
        self.records: deque = deque(maxlen=capacity)

    def emit(self, record: logging.LogRecord) -> None:
        ...

    def get_records(self) -> list[str]:
        ...


handler = InMemoryHandler(capacity=5)
handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
logger = logging.getLogger('mem')
logger.addHandler(handler)
logger.setLevel(logging.DEBUG)

for i in range(7):
    logger.info('Message %d', i)

print(handler.get_records())   # ostatnie 5 wiadomosci

---

## 3. 🔹 Loggery hierarchiczne i dobre praktyki

Loggery tworzą hierarchię według nazw rozdzielonych kropkami:
`app.db` jest dzieckiem `app`, które jest dzieckiem root loggera.

```
root
├── app
│   ├── app.db
│   └── app.api
└── utils
```

**Złota zasada bibliotek:** zawsze uzywaj `getLogger(__name__)`
i **nigdy** nie dodawaj handlerów w kodzie biblioteki - to zadanie
aplikacji. Dodaj tylko `NullHandler` by ciszyc ostrzezenia.

```python
# W bibliotece
logger = logging.getLogger(__name__)
logging.getLogger('mylib').addHandler(logging.NullHandler())

# W aplikacji - uzytkownik konfiguruje jak chce
logging.getLogger('mylib').setLevel(logging.DEBUG)
```

In [ ]:
import logging

# Hierarchia loggerow
logging.basicConfig(level=logging.DEBUG, format='%(name)s - %(levelname)s - %(message)s')

root_logger   = logging.getLogger()          # root
app_logger    = logging.getLogger('app')     # dziecko root
db_logger     = logging.getLogger('app.db')  # dziecko app
api_logger    = logging.getLogger('app.api') # dziecko app

app_logger.setLevel(logging.WARNING)   # app i jego dzieci: WARNING+

db_logger.debug('Query: SELECT *')     # nie wyswietli sie (poziom app=WARNING)
db_logger.warning('Slow query!')       # wyswietli sie
api_logger.info('GET /users')          # nie wyswietli sie
api_logger.error('500 Internal Error') # wyswietli sie

---

### 🐍 Ćwiczenia - hierarchia i dobre praktyki

**Ćw. 7.** Stwórz trzy loggery: `'service'`, `'service.db'`,
`'service.api'`. Ustaw poziom `WARNING` dla `'service'` i `DEBUG`
dla `'service.db'`. Sprawdź które komunikaty są widoczne dla każdego.

**Ćw. 8.** Napisz moduł-klasę `DatabaseService` używający
`logging.getLogger(__name__)` wewnętrznie. Metody: `connect()`,
`query(sql)`, `disconnect()`. Każda loguje na odpowiednim poziomie.

**Ćw. 9. *(Trudniejsze)*** Napisz kontekstowy filtr `RequestFilter`
dziedziczący po `logging.Filter`. Przechowuje `request_id` i dodaje
go do każdego rekordu jako atrybut. Uzyj `logger.addFilter()`
aby wszystkie logi w bloku zawierały ten sam `request_id`.

In [ ]:
# Ćw. 7
import logging

logging.basicConfig(level=logging.DEBUG, format='%(name)-15s %(levelname)s %(message)s')

service     = logging.getLogger('service')
service_db  = logging.getLogger('service.db')
service_api = logging.getLogger('service.api')

service.setLevel(...)
service_db.setLevel(...)

service_db.debug('DB debug')        # widoczny?
service_db.warning('DB warning')    # widoczny?
service_api.info('API info')        # widoczny?
service_api.error('API error')      # widoczny?

In [ ]:
# Ćw. 8
import logging

class DatabaseService:
    def __init__(self, dsn: str):
        self._logger = logging.getLogger(__name__)
        self._dsn = dsn

    def connect(self) -> None:
        ...

    def query(self, sql: str) -> list:
        ...

    def disconnect(self) -> None:
        ...


logging.basicConfig(level=logging.DEBUG, format='%(name)s - %(levelname)s - %(message)s')
db = DatabaseService('postgresql://localhost/mydb')
db.connect()
db.query('SELECT * FROM users')
db.disconnect()

In [ ]:
# Ćw. 9
import logging
import uuid

class RequestFilter(logging.Filter):
    def __init__(self, request_id: str):
        super().__init__()
        self.request_id = request_id

    def filter(self, record: logging.LogRecord) -> bool:
        # hint: dodaj atrybut record.request_id
        ...
        return True


logging.basicConfig(
    level=logging.DEBUG,
    format='[%(request_id)s] %(levelname)s %(message)s'
)
logger = logging.getLogger('api')

req_id = str(uuid.uuid4())[:8]
f = RequestFilter(req_id)
logger.addFilter(f)

logger.info('Przyjeto zadanie')
logger.info('Przetwarzanie...')
logger.info('Wysłano odpowiedz')

logger.removeFilter(f)